In [1]:
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import json
from pyscf import gto, scf, fci
import netket.experimental as nkx
import jax
import jax.numpy as jnp
from flax import nnx
import jax.tree_util as jtu
import sys
sys.path.insert(0, '.')


/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# H2分子几何构型
bond_length = 1.4  # 平衡键长（埃）
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

print("="*60)
print("H2分子系统设置")
print("="*60)
print(f"键长: {bond_length} 埃")

# 创建分子对象
mol = gto.M(atom=geometry, basis='STO-3G')

# Hartree-Fock计算
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"\nHartree-Fock能量: {E_hf:.8f} Ha")

# FCI计算（参考值）
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print(f"\nFCI能级（参考值）:")
print("-"*50)
for i, e in enumerate(E_fcis):
    exc_energy = (e - E_fcis[0]) * 27.2114
    if i == 0:
        print(f"  E{i} (基态)     = {e:.8f} Ha")
    else:
        print(f"  E{i} (第{i}激发态) = {e:.8f} Ha  激发能: {exc_energy:.4f} eV")

# 创建NetKet哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

H2分子系统设置
键长: 1.4 埃

Hartree-Fock能量: -0.94148065 Ha

FCI能级（参考值）:
--------------------------------------------------
  E0 (基态)     = -1.01546825 Ha
  E1 (第1激发态) = -0.87542794 Ha  激发能: 3.8107 eV
  E2 (第2激发态) = -0.42938376 Ha  激发能: 15.9482 eV
  E3 (第3激发态) = -0.26922131 Ha  激发能: 20.3064 eV


In [3]:
ha.to_dense()

Array([[-0.3432089 ,  0.        ,  0.        ,  0.22302209],
       [ 0.        , -0.65240585,  0.22302209,  0.        ],
       [ 0.        ,  0.22302209, -0.65240585,  0.        ],
       [ 0.22302209,  0.        ,  0.        , -0.94148065]],      dtype=float64)

In [4]:

def Ham_psi(ha:nk.operator.DiscreteOperator,model,x:jnp.array):
    x_primes, mels = ha.get_conn(x)
    psi_values = jax.vmap(model)(x_primes)
    H_psi_x = jnp.sum(mels*psi_values)
    return H_psi_x

In [5]:
# 创建Hilbert空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)

print("="*60)
print("Hilbert空间信息")
print("="*60)
print(f"空间轨道数: 2")
print(f"自旋轨道数: 4")
print(f"电子数: 2 (α=1, β=1)")
print(f"Hilbert空间维度: {hi.n_states}")
print(f"\n所有可能的电子组态:")
print(hi.all_states())

# 创建采样器
g = nk.graph.Graph(edges=[(0,1),(2,3)])
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=4, spin_symmetric=True, sweep_size=64
)

Hilbert空间信息
空间轨道数: 2
自旋轨道数: 4
电子数: 2 (α=1, β=1)
Hilbert空间维度: 4

所有可能的电子组态:
[[0 1 0 1]
 [0 1 1 0]
 [1 0 0 1]
 [1 0 1 0]]


In [23]:
class SingleStateAnsatz(nnx.Module):
    """单态 Ansatz：适配费米子系统的复数值 FFNN"""
    def __init__(self, n_spin_orbitals: int, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_spin_orbitals = n_spin_orbitals
        # 复数值线性层（费米子波函数需相位描述）
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x: jax.Array) -> jax.Array:
        """
        输入：x = 单组自旋轨道占据态（形状 [n_spin_orbitals,]，如 H₂ 的 [0,1,0,1]）
        输出：复数值波函数值 ψ(x)
        """
        h = nnx.tanh(self.linear1(x))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)  # 输出形状 []（标量复数）

In [24]:
model = SingleStateAnsatz(n_spin_orbitals=4,hidden_dim=4*3,rngs=nnx.Rngs(12))
# 2. 提取参数（旧版NNX专用）
ffnn_state= nnx.split(model)
def forward(state,x):
  y, ffnn_state = nnx.call(state)(x)
  return y

In [25]:
s = SingleStateAnsatz(n_spin_orbitals=4,hidden_dim=4*3,rngs=nnx.Rngs(1))
s([1,0,1,0])

Array(-0.20085351-1.00943396j, dtype=complex128)

In [26]:
class NESTotalAnsatz(nnx.Module):
    """NES-VMC 总 Ansatz：K 个单态 Ansatz 的行列式"""
    def __init__(self, n_spin_orbitals: int, n_states: int = 3, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_states = n_states  # K
        self.n_spin_orbitals = n_spin_orbitals
        
        # 初始化 K 个独立的单态 Ansatz
        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=nnx.Rngs(42+i))
            for i in range(n_states)
        ]

    def __call__(self, x_batch: jax.Array) -> tuple[jax.Array, jax.Array]:
        """x_batch.shape = [K, n_spin_orbitals]
        
        输出：
            psi_total : 行列式标量（给 NetKet 采样）
            M_matrix  : K×K 矩阵（给 Eq.17 局部能量计算）
        """
        if x_batch.shape[0] != self.n_states:
            raise ValueError(f"x_batch.shape[0] != {self.n_states}")
        K = self.n_states
        M = []
        for i in range(K):
            for j in range(K):
                # M[i,j] = ψ_j(x_i)
                psi_i_xj = self.single_ansatz_list[j](x_batch[i])
                M.append(psi_i_xj)
        
        # 构建 K×K 矩阵
        M = jnp.stack(M, axis=0)
        M_matrix = M.reshape(K, K)  # 这就是论文里的 Ψ 矩阵！
        # 计算行列式
        psi_total = jnp.linalg.det(M_matrix)
        return psi_total, M_matrix

In [27]:
# 测试环境：H₂ 系统（2 个轨道→4 个自旋轨道，K=2 个激发态）
n_spin_orbitals = 4
n_states = 2  # K=2
rngs = nnx.Rngs(seed=42)

# 初始化 Total Ansatz
total_ansatz = NESTotalAnsatz(
    n_spin_orbitals=n_spin_orbitals,
    n_states=n_states,
    hidden_dim=8,
    rngs=rngs
)

# 测试1：输入两组不同的状态（x1≠x2）
x1 = jnp.array([0,1,0,1])  # H₂ 基态自旋轨道占据态
x2 = jnp.array([1,0,1,0])  # 另一个不同状态
x_batch = jnp.stack([x1, x2], axis=0)  # 形状 [2,4]（K=2 组状态）
psi_valid,psi_matrix = total_ansatz(x_batch)
print(f"两组不同状态的行列式值：{psi_valid:.4f}")  # 非零（如 0.345+0.123j）

# 测试2：输入两组相同的状态（x1=x2，模拟态坍缩）
x_batch_collapse = jnp.stack([x1, x1], axis=0)  # 形状 [2,4]（两组状态相同）
psi_collapse,matrix = total_ansatz(x_batch_collapse)
print(f"两组相同状态的行列式值：{psi_collapse:.4f}")  # 接近 0（如 1e-6+2e-6j）

两组不同状态的行列式值：2.4032+3.0742j
两组相同状态的行列式值：0.0000+0.0000j


In [28]:
def Ham_Psi(ha:nk.operator.DiscreteOperator,total_ansatz,x:jnp.array):
    k = total_ansatz.n_states
    if x.shape[0] != k:
        raise ValueError(f"Input array must have shape ({k},) but got shape {x.shape}")
    H_psi_x_i = []
    for i in range(k):
        tmp = []
        for j in range(k):
            ele = Ham_psi(ha,model=total_ansatz.single_ansatz_list[j],x=x[i])
            tmp.append(ele)
            
    
        H_psi_x_i.append(tmp)
    
    HPsi = jnp.array(H_psi_x_i).reshape(k,k)    
    return HPsi            

In [29]:
k = Ham_Psi(ha,total_ansatz,x=x_batch)
k

Array([[-0.35336963+0.40729686j, -0.25753339+0.64114281j],
       [ 0.48052391-0.43010751j,  0.78080402-2.66369154j]],      dtype=complex128)

In [36]:
def compute_local_energy(ha, total_ansatz: NESTotalAnsatz, x_batch):
    psi, M = total_ansatz(x_batch)
    Hp = Ham_Psi(ha, total_ansatz, x_batch)
    el_mat = jnp.linalg.inv(M) @ Hp
    return jnp.trace(el_mat), el_mat


In [37]:
compute_local_energy(ha=ha,total_ansatz=total_ansatz,x_batch=x_batch)

(Array(-1.28468955+5.27355937e-16j, dtype=complex128),
 Array([[-0.39448102+0.01102113j, -0.37671114+0.2625076j ],
        [-0.14601127-0.08724348j, -0.89020853-0.01102113j]],      dtype=complex128))

In [38]:
# 把 total_ansatz 变成带参数的前向函数
def forward(params, x_batch):
    # 把参数绑定回模型
    nnx.update(total_ansatz, params)
    tr_el, _ = compute_local_energy(ha, total_ansatz, x_batch)
    return tr_el.real  # 能量取实部作为损失

In [ ]:
# 取模型参数
params = nnx.state(total_ansatz)
# 对参数求导（核心！）
loss, grads = jax.value_and_grad(forward)(params, x_batch)

In [40]:
def compute_local_energy(ha, total_ansatz: NESTotalAnsatz, x_batch):
    psi, M = total_ansatz(x_batch)
    Hp = Ham_Psi(ha, total_ansatz, x_batch)
    el_mat = jnp.linalg.inv(M) @ Hp
    energy = jnp.trace(el_mat)
    return energy.real  # 直接返回实值能量，方便求导

# 一键获取梯度
@jax.jit
def grad_fn(params, x_batch):
    nnx.update(total_ansatz, params)
    loss = compute_local_energy(ha, total_ansatz, x_batch)
    return loss, jax.grad(compute_local_energy, argnums=1)(ha, total_ansatz, x_batch)

In [42]:
n_orbitals = 2
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=n_orbitals,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)

g = nk.graph.Graph(edges=[(0,1), (2,3)])
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=4, spin_symmetric=True, sweep_size=64
)

In [41]:
# 损失：对一批样本平均能量
@jax.jit
def loss_fn(params, samples_batch):
    nnx.update(total_ansatz, params)
    total = 0.0
    for x in samples_batch:
        e = compute_local_energy(ha, total_ansatz, x)
        total += e
    return total / len(samples_batch)

# 梯度 + 损失 一起出
loss, grads = jax.value_and_grad(loss_fn)(params, samples_batch)

NameError: name 'samples_batch' is not defined